# Notebook 03: Clustering et Labellisation Faible

Ce notebook applique une analyse non supervisée pour générer des labels faibles à partir des features extraits.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage

DATA_DIR = Path('../data')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Imports terminés')

## 1. Chargement des features

In [ ]:
features_path = DATA_DIR / 'processed' / 'features_resnet50.npy'
if features_path.exists():
    features = np.load(features_path)
else:
    np.random.seed(42)
    features = np.random.rand(500, 2048)

true_labels = np.random.choice([0, 1], len(features), p=[0.7, 0.3])
print('Features shape:', features.shape)
print('Nombre de labels simulés:', len(true_labels))

## 2. Réduction de dimensionnalité

In [ ]:
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

pca_2d = PCA(n_components=2, random_state=42)
features_pca_2d = pca_2d.fit_transform(features_scaled)

pca_3d = PCA(n_components=3, random_state=42)
features_pca_3d = pca_3d.fit_transform(features_scaled)

tsne_2d = TSNE(n_components=2, random_state=42, perplexity=min(30, len(features)-1))
features_tsne_2d = tsne_2d.fit_transform(features_scaled)

print('PCA 2D:', features_pca_2d.shape)
print('PCA 3D:', features_pca_3d.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(features_pca_2d[:, 0], features_pca_2d[:, 1], alpha=0.6, s=15)
axes[0].set_title('PCA 2D')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(features_tsne_2d[:, 0], features_tsne_2d[:, 1], alpha=0.6, s=15)
axes[1].set_title('t-SNE 2D')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Choix du nombre de clusters

In [ ]:
def find_optimal_k(x, max_k=10):
    inertias = []
    silhouette_scores = []
    k_values = list(range(2, min(max_k + 1, len(x))))
    for k in k_values:
        model = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = model.fit_predict(x)
        inertias.append(model.inertia_)
        silhouette_scores.append(silhouette_score(x, labels))
    return k_values, inertias, silhouette_scores

k_values, inertias, silhouette_scores = find_optimal_k(features_pca_2d)
optimal_k = k_values[int(np.argmax(silhouette_scores))]
print('optimal_k =', optimal_k)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(k_values, inertias, 'o-')
axes[0].set_title('Méthode elbow')
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_values, silhouette_scores, 'o-')
axes[1].set_title('Score de silhouette')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Dendrogramme

In [ ]:
sample_size = min(100, len(features_pca_2d))
sample_idx = np.random.choice(len(features_pca_2d), sample_size, replace=False)
linkage_matrix = linkage(features_pca_2d[sample_idx], method='ward')

plt.figure(figsize=(12, 6))
dendrogram(linkage_matrix, truncate_mode='level', p=5)
plt.title('Dendrogramme')
plt.grid(True, alpha=0.3)
plt.show()

## 5. Application des algorithmes de clustering

In [ ]:
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(features_pca_2d)

hierarchical = AgglomerativeClustering(n_clusters=optimal_k)
hierarchical_labels = hierarchical.fit_predict(features_pca_2d)

dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(features_pca_2d)

print('K-Means:', len(np.unique(kmeans_labels)), 'clusters')
print('Hierarchical:', len(np.unique(hierarchical_labels)), 'clusters')
print('DBSCAN:', len(np.unique(dbscan_labels)), 'clusters')

In [ ]:
def evaluate_clustering_quality(x, labels, y_true=None):
    metrics = {}
    if len(np.unique(labels)) > 1:
        metrics['silhouette_score'] = float(silhouette_score(x, labels))
        metrics['davies_bouldin_score'] = float(davies_bouldin_score(x, labels))
    if y_true is not None and len(np.unique(labels)) > 1:
        metrics['adjusted_rand_score'] = float(adjusted_rand_score(y_true, labels))
    return metrics

kmeans_metrics = evaluate_clustering_quality(features_pca_2d, kmeans_labels, true_labels)
hierarchical_metrics = evaluate_clustering_quality(features_pca_2d, hierarchical_labels, true_labels)
dbscan_metrics = evaluate_clustering_quality(features_pca_2d, dbscan_labels, true_labels)

print('K-Means metrics:', kmeans_metrics)
print('Hierarchical metrics:', hierarchical_metrics)
print('DBSCAN metrics:', dbscan_metrics)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(features_pca_2d[:, 0], features_pca_2d[:, 1], c=kmeans_labels, cmap='viridis', s=12)
axes[0].set_title('K-Means')

axes[1].scatter(features_pca_2d[:, 0], features_pca_2d[:, 1], c=hierarchical_labels, cmap='plasma', s=12)
axes[1].set_title('Hiérarchique')

axes[2].scatter(features_pca_2d[:, 0], features_pca_2d[:, 1], c=dbscan_labels, cmap='coolwarm', s=12)
axes[2].set_title('DBSCAN')

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Génération des labels faibles

In [ ]:
weak_labels = kmeans_labels
weak_labels_path = DATA_DIR / 'processed' / 'weak_labels_kmeans.npy'
weak_labels_path.parent.mkdir(parents=True, exist_ok=True)
np.save(weak_labels_path, weak_labels)

clustering_results = {
    'algorithm': 'KMeans',
    'n_clusters': int(optimal_k),
    'metrics': kmeans_metrics,
    'labels': weak_labels.tolist()
}

clustering_results_path = RESULTS_DIR / 'clustering_results.json'
with open(clustering_results_path, 'w', encoding='utf-8') as f:
    json.dump(clustering_results, f, indent=2, ensure_ascii=False)

print('Labels faibles sauvegardés dans:', weak_labels_path)
print('Résultats sauvegardés dans:', clustering_results_path)